In [1]:
# --------------------------------------------------------
# 
# Read in the SA results, extract the model output metrics
# and plot 
# 
# --------------------------------------------------------

In [6]:
import glob
import os
import pickle
import re
import sys

import cmocean.cm as cmo
import fsspec         # for AWS integration
import s3fs
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from SALib.sample import morris as morris_sampler
from SALib.analyze import morris as morris_analyzer

# --- read in cdr calculation functions  
sys.path.append(os.path.abspath('/home/tykukla/ew-workflows/run_scepter'))
import sa_postproc_fxns as spf
# import cdr_fxns_postproc as cfp
# ---


# --- define the SA iteration (`setup/batch/SA_inputs/sa_dir`)
inputs_dir = '/home/tykukla/ew-workflows/scripts/scepter/setup/batch/SA_inputs'
sa_dir = 'SAmorris450_8_v0'

# --- set common filenames 
param_values_fn = "param_values.npy"
problem_fn = "problem.pkl"
cdr_dfs = "cdr_dfs_sum.pkl"

# --- where to save the resulting xr datasets
save_datasets = True
save_ds_here = f"/home/tykukla/ew-workflows/scripts/scepter/process/runs/batch_postproc_tmp/sa_morris/{sa_dir}_datasets"


In [3]:
# --- decide which results to read in 
outdir = "/home/tykukla/ew-workflows/scripts/scepter/process/runs/batch_postproc_tmp/sa_morris"
outputs_dict = {
    "cc_1yr": "meanAnn_shortRun_SAFert_cc_morris_8_450_1yrInt_001",
    "cc_5yr": "meanAnn_shortRun_SAFert_cc_morris_8_450_5yrInt_001",
    "gbas_1yr": "meanAnn_shortRun_SAFert_gbas_morris_8_450_1yrInt_001",
    "gbas_5yr": "meanAnn_shortRun_SAFert_gbas_morris_8_450_1yrInt_001",
}
# --- how to compare across cases
anom_outputs_dict = {
    "gbas_1yr - cc_1yr": True,    # whether to run this case 
    "gbas_5yr - cc_5yr": True,    # whether to run this case 
}
# --- plots to make 
plot_dict = {
    "plot_types": ["scatter", "scatter_mu", "bar"],
    "plot_metrics": ["cdr_dif", "cdr_adv", "fraction_remaining_dissolved"],
    "savepath": f"/home/tykukla/ew-workflows/scripts/scepter/process/scratch/sa_figures/{sa_dir}"
}
saveplots = False

In [4]:
# --- read in the problem and parameters
problem = pd.read_pickle(os.path.join(inputs_dir, sa_dir, problem_fn))
param_values = np.load(os.path.join(inputs_dir, sa_dir, param_values_fn))

In [5]:
# --- calculate sensitivity analysis and plot 
ds1 = spf.Si_plot_batch(
    outputs_dict=outputs_dict,
    outdir = outdir,
    cdr_dfs=cdr_dfs,
    problem = problem,
    param_values = param_values,
    saveplots = saveplots,
    plot_dict = plot_dict,
)

# repeat for anomaly sensitivity
ds2 = spf.Si_plot_batch_anoms(
    anom_outputs_dict = anom_outputs_dict,
    outputs_dict=outputs_dict,
    outdir = outdir,
    cdr_dfs=cdr_dfs,
    problem = problem,
    param_values = param_values,
    saveplots = saveplots,
    plot_dict = plot_dict,
)


Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True
Same order: True


In [7]:
# --- save ds1 and ds2
if save_datasets:
    os.makedirs(save_ds_here, exist_ok=True)
    ds1.to_netcdf(os.path.join(save_ds_here, 'cases.nc'))
    ds2.to_netcdf(os.path.join(save_ds_here, 'caseAnoms.nc'))

In [ ]:
# ------------------------